# 📖 Notebook 4: Advanced Temporal Patterns

Now that you know workflows, activities, and the Saga pattern, let's explore
advanced patterns that handle real-world complexity: breaking work into pieces,
waiting for human input, and scheduling delayed actions.

## Learning Objectives

By the end of this notebook, you'll understand:
- **Child Workflows** — break complex processes into smaller, independent workflows
- **Signals** — send data to a running workflow from the outside
- **Timers** — pause a workflow for a duration without consuming resources

## 🛠️ Setup

Make sure Temporal is running:

```bash
cd 03-technologies/workflow-engines/temporal
docker compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import asyncio
import uuid
from datetime import timedelta
from temporalio import activity, workflow
from temporalio.client import Client
from temporalio.worker import Worker
from temporalio.common import RetryPolicy

client = await Client.connect("localhost:7233")
print("✅ Connected to Temporal")

TASK_QUEUE = "advanced-task-queue"

---

## 🧒 Part 1: Child Workflows

### The Problem

Imagine you need to process a batch of 100 orders. You could put all the logic
in one giant workflow, but:
- The event history gets enormous (every step × 100 orders)
- You can't retry a single order independently
- You can't see the status of individual orders

### The Solution: Child Workflows

A **parent workflow** spawns **child workflows** — each child is a fully independent
workflow with its own event history, retry behavior, and status.

```
┌───────────────────────────────────┐
│  BatchWorkflow (parent)           │
│                                   │
│  ┌───────────────────────────┐    │
│  │ ProcessItem (child 1)     │    │
│  └───────────────────────────┘    │
│  ┌───────────────────────────┐    │
│  │ ProcessItem (child 2)     │    │
│  └───────────────────────────┘    │
│  ┌───────────────────────────┐    │
│  │ ProcessItem (child 3)     │    │
│  └───────────────────────────┘    │
└───────────────────────────────────┘
```

Each child appears as its own workflow in the Temporal UI.

In [ ]:
# Activities for our batch processing example

@activity.defn
async def process_item(item: str) -> dict:
    """Process a single item — simulates some real work."""
    activity.logger.info(f"Processing item: {item}")
    await asyncio.sleep(0.5)
    return {"item": item, "status": "processed", "result": f"Done-{item}"}


@activity.defn
async def summarize_results(results: list) -> dict:
    """Summarize all results from the batch."""
    activity.logger.info(f"Summarizing {len(results)} results")
    await asyncio.sleep(0.2)
    return {
        "total": len(results),
        "succeeded": sum(1 for r in results if r["status"] == "processed"),
    }


print("✅ Defined activities: process_item, summarize_results")

In [ ]:
# The Child Workflow — processes a single item independently

@workflow.defn
class ProcessItemWorkflow:
    """Child workflow: processes one item with its own retry and history."""

    @workflow.run
    async def run(self, item: str) -> dict:
        result = await workflow.execute_activity(
            process_item,
            item,
            start_to_close_timeout=timedelta(seconds=30),
            retry_policy=RetryPolicy(maximum_attempts=3),
        )
        return result


# The Parent Workflow — spawns child workflows for each item

@workflow.defn
class BatchWorkflow:
    """Parent workflow: processes a batch of items using child workflows."""

    @workflow.run
    async def run(self, items: list) -> dict:
        workflow.logger.info(f"Starting batch of {len(items)} items")

        # Launch all child workflows concurrently!
        child_handles = []
        for item in items:
            handle = await workflow.start_child_workflow(
                ProcessItemWorkflow.run,
                item,
                id=f"process-{item}-{workflow.info().workflow_id}",
            )
            child_handles.append(handle)

        # Wait for all children to complete
        results = []
        for handle in child_handles:
            result = await handle
            results.append(result)

        # Summarize results
        summary = await workflow.execute_activity(
            summarize_results,
            results,
            start_to_close_timeout=timedelta(seconds=10),
        )

        return {"summary": summary, "results": results}


print("✅ Defined ProcessItemWorkflow (child) and BatchWorkflow (parent)")
print()
print("Key points:")
print("  • workflow.start_child_workflow() launches a child and returns a handle")
print("  • await handle gets the child's result when it completes")
print("  • Children run concurrently — all 3 items process in parallel")

In [ ]:
# Run the batch workflow

async def run_batch():
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[BatchWorkflow, ProcessItemWorkflow],
        activities=[process_item, summarize_results],
    ):
        return await client.execute_workflow(
            BatchWorkflow.run,
            ["item-A", "item-B", "item-C"],
            id=f"batch-{uuid.uuid4()}",
            task_queue=TASK_QUEUE,
        )


result = await run_batch()

print("📦 Batch Results:")
print(f"   Total items:  {result['summary']['total']}")
print(f"   Succeeded:    {result['summary']['succeeded']}")
print()
print("📋 Individual Results:")
for r in result["results"]:
    print(f"   {r['item']}: {r['status']} → {r['result']}")
print()
print("💡 Check the Temporal UI — you'll see the parent workflow AND each")
print("   child workflow as separate entries with their own event histories!")

### When to Use Child Workflows

| Use Case | Why Children Help |
|----------|------------------|
| Batch processing | Each item has independent retry and history |
| Different retry policies | Parent and children can have different settings |
| Event history size | Each child has a smaller, manageable history |
| Independent visibility | Monitor each sub-task separately in the UI |
| Reusability | The same child workflow can be used by different parents |

---

## 📡 Part 2: Signals

### The Problem

Sometimes a workflow needs to **wait for external input** before continuing:
- A manager needs to **approve** an expense report
- A user needs to **verify** their email address
- A compliance team needs to **review** a large transaction

With cron + DB, you'd poll a table: `SELECT * FROM approvals WHERE order_id = ?`

### The Solution: Signals

A **signal** is a message sent to a running workflow from the outside.
The workflow can pause and wait for a signal, then continue when it arrives.

```
Workflow starts         Signal arrives          Workflow continues
     │                       │                       │
     ▼                       ▼                       ▼
┌─────────┐  wait...  ┌──────────┐  resume  ┌─────────────┐
│ Process  │─────────▶│ Waiting  │─────────▶│ Continue    │
│ order    │          │ for      │          │ with        │
│          │          │ approval │          │ shipping    │
└─────────┘          └──────────┘          └─────────────┘
```

While waiting, the workflow **consumes zero resources**. Temporal handles the wait.

In [ ]:
# Activities for the approval workflow

@activity.defn
async def submit_for_review(order_id: str) -> dict:
    """Submit an order for manager review."""
    activity.logger.info(f"Order {order_id} submitted for manager review")
    await asyncio.sleep(0.3)
    return {"order_id": order_id, "review_status": "pending"}


@activity.defn
async def fulfill_order(order_id: str) -> dict:
    """Fulfill an approved order."""
    activity.logger.info(f"Fulfilling approved order {order_id}")
    await asyncio.sleep(0.5)
    return {"order_id": order_id, "status": "fulfilled"}


@activity.defn
async def reject_order(order_id: str) -> dict:
    """Handle a rejected order."""
    activity.logger.info(f"Order {order_id} was rejected")
    await asyncio.sleep(0.2)
    return {"order_id": order_id, "status": "rejected"}


print("✅ Defined activities: submit_for_review, fulfill_order, reject_order")

In [ ]:
# Workflow that waits for a signal (manager approval)

@workflow.defn
class ApprovalWorkflow:
    """
    An order workflow that pauses for human approval.

    The workflow:
    1. Submits the order for review
    2. WAITS for an approval signal (from a manager)
    3. Proceeds based on the decision: fulfill or reject
    """

    def __init__(self):
        self.approved = None       # None = waiting, True = approved, False = rejected
        self.approver_name = None  # who approved/rejected

    @workflow.signal
    async def approve(self, approver: str):
        """Signal: a manager approved the order."""
        self.approved = True
        self.approver_name = approver

    @workflow.signal
    async def reject(self, approver: str):
        """Signal: a manager rejected the order."""
        self.approved = False
        self.approver_name = approver

    @workflow.run
    async def run(self, order_id: str) -> dict:
        # Step 1: Submit for review
        await workflow.execute_activity(
            submit_for_review, order_id,
            start_to_close_timeout=timedelta(seconds=10),
        )

        # Step 2: WAIT for a signal (approve or reject)
        # This line pauses the workflow — it uses ZERO resources while waiting!
        await workflow.wait_condition(lambda: self.approved is not None)

        # Step 3: Act on the decision
        if self.approved:
            result = await workflow.execute_activity(
                fulfill_order, order_id,
                start_to_close_timeout=timedelta(seconds=30),
            )
            return {
                "status": "fulfilled",
                "approved_by": self.approver_name,
                "result": result,
            }
        else:
            result = await workflow.execute_activity(
                reject_order, order_id,
                start_to_close_timeout=timedelta(seconds=10),
            )
            return {
                "status": "rejected",
                "rejected_by": self.approver_name,
                "result": result,
            }


print("✅ Defined ApprovalWorkflow with signals: approve, reject")
print()
print("Key points:")
print("  • @workflow.signal defines a method that can be called from outside")
print("  • workflow.wait_condition() pauses until a condition is True")
print("  • While waiting, the workflow uses ZERO compute resources")

In [ ]:
# Demo: Start the workflow, then send an approval signal

signal_activities = [submit_for_review, fulfill_order, reject_order]


async def demo_approval():
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[ApprovalWorkflow],
        activities=signal_activities,
    ):
        # Start the workflow (it will pause at wait_condition)
        workflow_id = f"approval-{uuid.uuid4()}"
        handle = await client.start_workflow(
            ApprovalWorkflow.run,
            "ORD-VIP-001",
            id=workflow_id,
            task_queue=TASK_QUEUE,
        )
        print(f"📋 Workflow started: {workflow_id}")
        print("   Status: waiting for manager approval...")

        # Simulate a manager reviewing and approving after 2 seconds
        await asyncio.sleep(2)
        print("\n👩‍💼 Manager 'Alice' sends APPROVE signal...")
        await handle.signal(ApprovalWorkflow.approve, "Alice")

        # Wait for the workflow to complete
        result = await handle.result()
        return result


result = await demo_approval()

print(f"\n📬 Workflow Result:")
print(f"   Status:      {result['status']}")
print(f"   Approved by: {result['approved_by']}")
print()
print("💡 The workflow was PAUSED waiting for a human decision.")
print("   It could have waited minutes, hours, or even days!")
print("   No polling, no cron jobs, no wasted resources.")

In [ ]:
# Demo: What about rejection?

async def demo_rejection():
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[ApprovalWorkflow],
        activities=signal_activities,
    ):
        workflow_id = f"rejection-{uuid.uuid4()}"
        handle = await client.start_workflow(
            ApprovalWorkflow.run,
            "ORD-SUSPICIOUS-001",
            id=workflow_id,
            task_queue=TASK_QUEUE,
        )
        print(f"📋 Workflow started: {workflow_id}")
        print("   Status: waiting for manager approval...")

        # Manager rejects this one
        await asyncio.sleep(1)
        print("\n👨‍💼 Manager 'Bob' sends REJECT signal...")
        await handle.signal(ApprovalWorkflow.reject, "Bob")

        result = await handle.result()
        return result


result = await demo_rejection()

print(f"\n📬 Workflow Result:")
print(f"   Status:      {result['status']}")
print(f"   Rejected by: {result['rejected_by']}")

### Real-World Signal Use Cases

| Use Case | Signal | What Happens |
|----------|--------|-------------|
| Expense approval | `approve(manager_id)` | Workflow submits expense for reimbursement |
| Email verification | `email_verified()` | Workflow activates the user account |
| Payment confirmation | `payment_received(amount)` | Workflow ships the order |
| Manual override | `skip_step(reason)` | Workflow skips a failing step |
| Order cancellation | `cancel(reason)` | Workflow runs compensation and stops |

---

## ⏰ Part 3: Timers

### The Problem

Many business processes involve **waiting**:
- Send a reminder email if the user doesn't verify within 24 hours
- Escalate a support ticket if not resolved within 4 hours
- Renew a subscription every 30 days
- Cancel an unpaid order after 48 hours

With cron, you'd scan a table every minute looking for items that need action.
This is wasteful and imprecise.

### The Solution: workflow.sleep()

Temporal workflows can sleep for any duration — seconds, hours, or even months.
While sleeping, the workflow uses **zero compute resources**. Temporal wakes it
up at exactly the right time.

```python
# This is real code — the workflow sleeps for 30 days!
await workflow.sleep(timedelta(days=30))
```

In [ ]:
# Activities for our timer examples

@activity.defn
async def send_reminder_email(user_id: str) -> str:
    """Send a reminder email to the user."""
    activity.logger.info(f"Sending reminder email to {user_id}")
    await asyncio.sleep(0.2)
    return f"Reminder sent to {user_id}"


@activity.defn
async def cancel_pending_order(order_id: str) -> dict:
    """Cancel an order that was never paid."""
    activity.logger.info(f"Cancelling unpaid order {order_id}")
    await asyncio.sleep(0.3)
    return {"order_id": order_id, "status": "cancelled", "reason": "payment_timeout"}


@activity.defn
async def confirm_order(order_id: str) -> dict:
    """Confirm a paid order."""
    activity.logger.info(f"Confirming paid order {order_id}")
    await asyncio.sleep(0.3)
    return {"order_id": order_id, "status": "confirmed"}


print("✅ Defined activities: send_reminder_email, cancel_pending_order, confirm_order")

In [ ]:
# Workflow that uses a timer for payment deadline
#
# Real scenario: Customer has a limited time to pay.
# If they don't pay within the timeout, cancel the order.
# If they pay (via signal), confirm the order.
#
# For the demo we use 5 seconds instead of 48 hours.

@workflow.defn
class PaymentDeadlineWorkflow:
    """
    Wait for payment within a deadline. If payment isn't received
    in time, cancel the order. Uses both a timer AND a signal.
    """

    def __init__(self):
        self.paid = False

    @workflow.signal
    async def payment_received(self):
        """Signal: the customer has paid."""
        self.paid = True

    @workflow.run
    async def run(self, order_id: str) -> dict:
        # Wait for EITHER:
        #   a) Payment signal arrives (self.paid becomes True)
        #   b) 5 seconds pass (timeout)
        #
        # In production, you'd use timedelta(hours=48) instead.
        payment_deadline = timedelta(seconds=5)

        try:
            # wait_condition with a timeout — this is the key pattern!
            await workflow.wait_condition(
                lambda: self.paid,
                timeout=payment_deadline,
            )
        except asyncio.TimeoutError:
            # Timer expired — customer didn't pay in time
            pass

        if self.paid:
            result = await workflow.execute_activity(
                confirm_order, order_id,
                start_to_close_timeout=timedelta(seconds=10),
            )
            return {"status": "confirmed", "result": result}
        else:
            result = await workflow.execute_activity(
                cancel_pending_order, order_id,
                start_to_close_timeout=timedelta(seconds=10),
            )
            return {"status": "cancelled", "reason": "payment_timeout", "result": result}


print("✅ Defined PaymentDeadlineWorkflow")
print("   Combines a timer (5s deadline) with a signal (payment_received)")

In [ ]:
# Demo 1: Customer pays before the deadline

timer_activities = [send_reminder_email, cancel_pending_order, confirm_order]


async def demo_pays_on_time():
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[PaymentDeadlineWorkflow],
        activities=timer_activities,
    ):
        workflow_id = f"deadline-pays-{uuid.uuid4()}"
        handle = await client.start_workflow(
            PaymentDeadlineWorkflow.run,
            "ORD-TIMELY",
            id=workflow_id,
            task_queue=TASK_QUEUE,
        )
        print(f"📋 Order placed. Payment deadline: 5 seconds")
        print("   Waiting for payment...")

        # Customer pays after 2 seconds (before the 5s deadline)
        await asyncio.sleep(2)
        print("   💳 Customer pays after 2 seconds!")
        await handle.signal(PaymentDeadlineWorkflow.payment_received)

        result = await handle.result()
        return result


result = await demo_pays_on_time()
print(f"\n📬 Result: {result['status']}")
print("✅ Customer paid on time — order confirmed!")

In [ ]:
# Demo 2: Customer doesn't pay — deadline expires

async def demo_payment_timeout():
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[PaymentDeadlineWorkflow],
        activities=timer_activities,
    ):
        workflow_id = f"deadline-timeout-{uuid.uuid4()}"
        handle = await client.start_workflow(
            PaymentDeadlineWorkflow.run,
            "ORD-LATE",
            id=workflow_id,
            task_queue=TASK_QUEUE,
        )
        print(f"📋 Order placed. Payment deadline: 5 seconds")
        print("   Waiting for payment...")
        print("   ⏳ (customer doesn't pay)")

        # Don't send any signal — let the deadline expire
        result = await handle.result()
        return result


result = await demo_payment_timeout()
print(f"\n📬 Result: {result['status']} — {result['reason']}")
print("⏰ Deadline expired — order was automatically cancelled!")
print()
print("💡 In production, this deadline would be hours or days.")
print("   Temporal handles the wait efficiently — no polling needed.")

### How Timers Work Under the Hood

When your workflow calls `workflow.sleep()` or `wait_condition(timeout=...)`,
Temporal:

1. **Records a TimerStarted event** in the workflow history
2. **Releases the worker** — no thread or process is blocked
3. **Schedules a wake-up** in its internal timer system
4. When the timer fires, **records TimerFired event**
5. **Schedules a workflow task** for the worker to continue

This means a workflow sleeping for 30 days uses **zero CPU and zero memory**
during that time. You can have millions of sleeping workflows.

---

## 🔧 Combining Patterns

In real systems, you combine these patterns. Here's a realistic order workflow:

```
OrderWorkflow (parent)
  │
  ├── Activity: validate_order()
  │
  ├── Child Workflow: PaymentWorkflow
  │     ├── Activity: charge_card()
  │     └── Timer: wait 48h for bank confirmation
  │
  ├── Signal: wait for fraud_review_complete
  │
  ├── Child Workflow: FulfillmentWorkflow
  │     ├── Activity: reserve_inventory()
  │     ├── Activity: schedule_shipping()
  │     └── Saga: compensate on failure
  │
  └── Activity: send_confirmation_email()
```

Each pattern solves a specific problem:
- **Child workflows** break complex logic into manageable pieces
- **Signals** handle human-in-the-loop decisions
- **Timers** manage deadlines and scheduled actions
- **Sagas** ensure consistency when steps fail

## 📚 Summary

### What We Learned

| Pattern | When to Use | Key API |
|---------|------------|----------|
| **Child Workflows** | Break up complex processes, batch processing | `workflow.start_child_workflow()` |
| **Signals** | External input, human approval, inter-workflow communication | `@workflow.signal` + `workflow.wait_condition()` |
| **Timers** | Deadlines, scheduled actions, delayed processing | `workflow.sleep()` or `wait_condition(timeout=...)` |

### Key Takeaways

1. **Child workflows** give each sub-task its own event history, retry behavior, and visibility
2. **Signals** let running workflows receive data from the outside world
3. **Timers** are resource-free — millions of sleeping workflows cost nothing
4. **Combine patterns** freely — children + signals + timers + sagas work together
5. **The Temporal UI** lets you inspect all of this visually

### What's Next?

You now have a solid foundation in Temporal! Here are some directions to explore:

- **Worker/**: Check out the `worker/` directory in this lab for a production-style worker setup
- **Temporal docs**: https://docs.temporal.io/develop/python
- **Testing**: Temporal has a built-in test framework — `temporalio.testing.WorkflowEnvironment`
- **Versioning**: Safely update workflow code while old workflows are still running
- **Observability**: Connect Temporal to Prometheus/Grafana for production monitoring